In [1]:
import random
import requests
from datetime import datetime
import time
import os
from dotenv import load_dotenv
import pandas as pd
load_dotenv()

ACCOUNT = os.getenv('STOCK_ACCOUNT_USERNAME')
PASSWORD = os.getenv('STOCK_ACCOUNT_PASSWORD')
PG_URI = os.getenv('POSTGRES_URI')

In [2]:
from util_stock_info import *
Get_Stock_Informations(2330, "20260316", "20260316")

[{'date': 1773619200, 'capacity': '35017289.00000', 'turnover': '65021963085.00000', 'high': '1880.00000', 'low': '1845.00000', 'close': '1845.00000', 'change': '-20.00000', 'transaction_volume': 221241, 'stock_code_id': '2330', 'open': '1875.00000'}]


[{'date': 1773619200,
  'capacity': '35017289.00000',
  'turnover': '65021963085.00000',
  'high': '1880.00000',
  'low': '1845.00000',
  'close': '1845.00000',
  'change': '-20.00000',
  'transaction_volume': 221241,
  'stock_code_id': '2330',
  'open': '1875.00000'}]

In [3]:
Get_User_Stocks(ACCOUNT, PASSWORD)

[]


[]

## 01 Stock overview

In [4]:
stock_info = pd.read_json('http://140.116.86.242:8081/api/stock/get_stock_list')
stock_info.head()

,result,data
0,success,"{'code': '0050', 'name': '元大台灣50', 'listing_ti..."
1,success,"{'code': '0051', 'name': '元大中型100', 'listing_t..."
2,success,"{'code': '0052', 'name': '富邦科技', 'listing_time..."
3,success,"{'code': '0053', 'name': '元大電子', 'listing_time..."
4,success,"{'code': '0054', 'name': '元大台商50', 'listing_ti..."


In [5]:
pd.json_normalize(stock_info['data'][0])

,code,name,listing_time,industry,type
0,0050,元大台灣50,1056902400,,0


In [6]:
from FinMind.data import DataLoader

api = DataLoader()
# api.login_by_token(api_token='token')
df = api.taiwan_stock_info_with_warrant()
df.head()

2026-03-29 05:30:14.142 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-03-29 05:30:14.143 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockInfoWithWarrant, data_id: 


,industry_category,stock_id,stock_name,type,date
0,封閉式基金,0001,鴻運,twse,2024-12-05
1,封閉式基金,0002,福元,twse,2024-12-05
2,封閉式基金,0003,成長,twse,2024-12-05
3,封閉式基金,0004,國民,twse,2024-12-05
4,封閉式基金,0005,成功,twse,2024-12-05


In [7]:
print(df['industry_category'].unique())

['封閉式基金' 'ETF' '上櫃ETF' '上櫃指數股票型基金(ETF)' '受益證券' 'ETN' '指數投資證券(ETN)'
 '認購權證(不含牛證)' '熊證(不含可展延熊證)' '牛證(不含可展延牛證)' '認售權證(不含熊證)' '可展延牛證' '水泥工業' '其他'
 '食品工業' '電器電纜' '農業科技業' '觀光事業' '觀光餐旅' '塑膠工業' '建材營造' '汽車工業' '電子零組件類' '紡織纖維'
 '貿易百貨' '運動休閒' '電子工業' '電子零組件業' '電機機械' '可轉換公司債' '生技醫療類' '電腦及週邊類' '運動休閒類'
 '化學生技醫療' '生技醫療業' '化學工業' '其他電子類' '玻璃陶瓷' '造紙工業' '鋼鐵工業' '居家生活' '綠能環保' '橡膠工業'
 '航運業' '創新板股票' '創新版股票' '電腦及週邊設備業' '半導體業' '其他電子業' '通信網路業' '光電業' '電子通路業'
 '資訊服務業' '油電燃氣業' '數位雲端類' '金融保險' '居家生活類' '文化創意業' '光電業類' '半導體類' '綠能環保類'
 '通信網路類' '電子商務業' '數位雲端' '資訊服務類' '電子通路類' '金融業' '認購售權證' '牛熊證(不含展延型牛熊證)'
 '油電燃氣類' '存託憑證']


In [8]:
# df.to_sql(name='stock_info', con=PG_URI, if_exists='replace', index=False)

### 01-1 Filter out low liquidity stocks
- Focus on twse type stocks
- Focus on normal industry stocks and ETFs
- Remove the stocks with low transaction volumes (Use 2026 Feb monthly transaction volume) 
(https://openapi.twse.com.tw/v1/exchangeReport/FMSRFK_ALL)

In [9]:
target_categories = [
    'ETF', '水泥工業', '食品工業', 
    '電器電纜', '農業科技業', '觀光餐旅', '塑膠工業', '建材營造', 
    '汽車工業', '電子零組件業', '紡織纖維', '貿易百貨', '運動休閒', 
    '電子工業', '電機機械', '生技醫療業', '電腦及週邊設備業', '化學工業', 
    '其他電子業', '玻璃陶瓷', '造紙工業', '鋼鐵工業', '居家生活', 
    '橡膠工業', '航運業', '半導體業', '通信網路業', '光電業', 
    '電子通路業', '資訊服務業', '油電燃氣業', '數位雲端', '金融保險', 
    '文化創意業', '綠能環保', '電子商務業'
]
target_type = ['twse']
filter_df = df[df['industry_category'].isin(target_categories) & df['type'].isin(target_type)]
print(df.shape, filter_df.shape)

(127161, 5) (1964, 5)


In [10]:
# Filter out stocks with low liquidity
def filter_liquidity(trade_vol_share: int = 100*1000, trade_value_ntd: int = 5000000) -> pd.DataFrame:
    """
    This function filters out stocks that do not meet the specified liquidity criteria.
    Stocks must have a daily trading volume greater than `trade_vol_share` shares or a daily
    trading value greater than `trade_value_ntd` NTD.   
    
    The data is fetched from the Taiwan Stock Exchange (TWSE) API(上市個股月成交資訊), which provides the last month's trading volume 
    and value for all listed stocks. The function processes this data to identify stocks that meet the liquidity 
    requirements and returns a DataFrame containing the filtered stocks.
    """
    # Fetch data and preprocess
    try:
        month_stock_vol = pd.read_json('https://openapi.twse.com.tw/v1/exchangeReport/FMSRFK_ALL')
        month_stock_vol.columns = month_stock_vol.columns.str.lower()
        print(f"Original shape: {month_stock_vol.shape}")

        filter_vol = month_stock_vol['tradevolumeb'] >= trade_vol_share
        filter_value = month_stock_vol['tradevaluea'] >= trade_value_ntd
        filtered_stocks = month_stock_vol[filter_vol & filter_value]
        print(f"Filtered shape: {filtered_stocks.shape}")
        return filtered_stocks
    except Exception as e:
        print(f"Error fetching or processing data: {e}")
        return pd.DataFrame()
    
filtered_stocks = filter_liquidity()
# filtered_stocks.to_sql(name='high_liquidity_stocks', con=PG_URI, if_exists='replace', index=False)
filtered_stocks.head()

Original shape: (28238, 10)
Filtered shape: (3198, 10)


,month,code,name,highestprice,lowestprice,weightedavgpriceab,transaction,tradevaluea,tradevolumeb,turnoverratio
0,11502,0050,元大台灣50,81.80,70.80,75.80,1652511,122099599220,1610793275,9.79
1,11502,0051,元大中型100,113.85,101.15,106.94,7688,155465337,1453748,6.05
2,11502,0052,富邦科技,48.85,42.09,45.34,400720,30854128818,680495910,30.47
3,11502,0053,元大電子,180.50,156.00,166.65,2657,55146058,330895,6.63
4,11502,0055,元大MSCI金融,34.81,31.60,33.57,5798,121266315,3611623,4.50


In [11]:
final_df = filter_df[filter_df['stock_id'].isin(filtered_stocks['code'])].reset_index(drop=True)
final_df['date'] = pd.to_datetime(final_df['date']).dt.date
# final_df.to_sql(name='targeted_stock_info', con=PG_URI, if_exists='replace', index=False)
final_df.head()

,industry_category,stock_id,stock_name,type,date
0,ETF,0050,元大台灣50,twse,2026-03-28
1,ETF,0051,元大中型100,twse,2026-03-28
2,ETF,0052,富邦科技,twse,2026-03-28
3,ETF,0053,元大電子,twse,2026-03-28
4,ETF,0055,元大MSCI金融,twse,2026-03-28


In [12]:
final_df.value_counts('industry_category').sort_values(ascending=False).reset_index(name='count').head(10)

,industry_category,count
0,電子工業,453
1,ETF,209
2,電子零組件業,102
3,半導體業,94
4,光電業,70
5,電腦及週邊設備業,66
6,生技醫療業,58
7,建材營造,55
8,電機機械,50
9,金融保險,49


### 01-2 Filter transaction value top 5 stocks of each industry

In [13]:
mapping_df = df[['stock_id', 'industry_category']]
filtered_stocks_n = filtered_stocks.merge(mapping_df, left_on='code', right_on='stock_id', how='left')
filtered_stocks_n['tradevaluea'] = pd.to_numeric(filtered_stocks_n['tradevolumeb'], errors='coerce')

# Select top 5 stocks by trading volume for each industry category
top5_stocks = filtered_stocks_n.groupby('industry_category', group_keys=False).apply(lambda x: x.nlargest(5, 'tradevolumeb'))

# Display the results (sorted by industry for easier viewing)
top5_stocks[['code', 'name', 'industry_category', 'tradevaluea']].sort_values(['industry_category', 'tradevaluea'], ascending=[True, False])

# Get the top 5 list
inspect_categories = ['建材營造', '電子零組件業', '半導體業', '通信網路業']
# inspect_categories = ['ETF',  '建材營造', '電子零組件業', '半導體業', '通信網路業']
# inspect_categories = ['ETF']
top5_stocks_code = top5_stocks[top5_stocks['industry_category'].isin(inspect_categories)]['code'].tolist()
print(top5_stocks_code)

['6770', '2337', '2344', '2303', '2408', '2515', '5521', '2542', '1316', '2442', '2485', '6285', '2455', '2412', '4977', '2367', '2313', '3037', '4958', '2327']


/tmp/ipykernel_1266/2560183240.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top5_stocks = filtered_stocks_n.groupby('industry_category', group_keys=False).apply(lambda x: x.nlargest(5, 'tradevolumeb'))


## 02 Fetch stock daily

In [18]:
# Create Date list
START_DATE = datetime.strptime("2025-02-01", "%Y-%m-%d")
END_DATE = datetime.strptime("2026-02-23", "%Y-%m-%d")

# 使用 'MS' (Month Start) 頻率來取得每個月的第一天
date_list = pd.date_range(start=START_DATE, end=END_DATE, freq='MS').strftime("%Y%m%d").tolist()
date_list

['20250201',
 '20250301',
 '20250401',
 '20250501',
 '20250601',
 '20250701',
 '20250801',
 '20250901',
 '20251001',
 '20251101',
 '20251201',
 '20260101',
 '20260201']

In [17]:
Get_Stock_Daily_Information('2330', '20251201')

,日期,成交股數,成交金額,開盤價,最高價,最低價,收盤價,漲跌價差,成交筆數,註記,股票代號
0,2025-12-01,33722188,47998505254,1445.0,1445.0,1410.0,1410.0,-30.0,102798,,2330
1,2025-12-02,27923064,39916596995,1430.0,1440.0,1420.0,1430.0,20.0,34886,,2330
2,2025-12-03,25462840,36775822053,1440.0,1450.0,1435.0,1450.0,20.0,32546,,2330
3,2025-12-04,21934952,31600541870,1445.0,1450.0,1430.0,1445.0,-5.0,44721,,2330
4,2025-12-05,22539737,32677962256,1445.0,1460.0,1440.0,1460.0,15.0,35327,,2330
5,2025-12-08,27405663,40634878504,1465.0,1495.0,1460.0,1495.0,35.0,65198,,2330
6,2025-12-09,27555343,40931363559,1495.0,1500.0,1480.0,1480.0,-15.0,47501,,2330
7,2025-12-10,26387641,39493603134,1490.0,1505.0,1485.0,1505.0,25.0,53469,,2330
8,2025-12-11,31820417,47234941277,1510.0,1515.0,1465.0,1470.0,NaN,107208,,2330
9,2025-12-12,24025183,35489107425,1475.0,1485.0,1470.0,1480.0,10.0,41662,,2330


In [26]:
from tqdm import tqdm
temp_list = []
# top3_stocks_code = ['009816', '0050', '00919']
# 使用正確的迴圈邏輯：日期 -> 股票代码
for code in top5_stocks_code:
    for date in tqdm(date_list, desc=f"Fetching data for {code}", total=len(date_list)):
        # print(f"Fetching {stock_code} on {top3_stocks_code}")
        tmp_df = Get_Stock_Daily_Information(code, date)
        # file_name = f"{code}_{date}.csv"
        # path = os.path.join("/app/data", file_name)
        # tmp_df.to_csv(path, index=False) # Save to CSV for backup
        if not tmp_df.empty:
            temp_list.append(tmp_df)
        time.sleep(3) # Global sleep to be safe
    concat_df = pd.concat(temp_list, ignore_index=True)
    concat_df.to_sql(name=f'daily_info_{code}', con=PG_URI, if_exists='replace', index=False)
    temp_list = []
concat_df.head()

Fetching data for 2327: 100%|██████████| 13/13 [01:20<00:00,  6.18s/it]


,日期,成交股數,成交金額,開盤價,最高價,最低價,收盤價,漲跌價差,成交筆數,註記,股票代號
0,2025-02-03,2978083,1544736010,529.0,529.0,513.0,519.0,-18.0,5406,,2327
1,2025-02-04,3316001,1769930292,526.0,548.0,526.0,534.0,15.0,4124,,2327
2,2025-02-05,9157452,5155738217,541.0,576.0,541.0,569.0,35.0,12411,,2327
3,2025-02-06,8821955,5107451200,578.0,591.0,569.0,571.0,2.0,17975,,2327
4,2025-02-07,3103063,1766604379,572.0,574.0,566.0,571.0,0.0,9758,,2327


## END